In [156]:
import pandas as pd
df=pd.read_csv(r"C:\Users\RAJAT ANAND\Downloads\retail_sales_dataset.csv")
df

,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount
0,1,24-11-2023,CUST001,Male,34,Beauty,3,50,150
1,2,27-02-2023,CUST002,Female,26,Clothing,2,500,1000
2,3,13-01-2023,CUST003,Male,50,Electronics,1,30,30
3,4,21-05-2023,CUST004,Male,37,Clothing,1,500,500
4,5,06-05-2023,CUST005,Male,30,Beauty,2,50,100
...,...,...,...,...,...,...,...,...,...
995,996,16-05-2023,CUST996,Male,62,Clothing,1,50,50
996,997,17-11-2023,CUST997,Male,52,Beauty,3,30,90
997,998,29-10-2023,CUST998,Female,23,Beauty,4,25,100
998,999,05-12-2023,CUST999,Female,36,Electronics,3,50,150


In [157]:
df["Date"]=pd.to_datetime(df["Date"],dayfirst= True)

In [158]:
df["revenue"]=df["Quantity"]*df["Price per Unit"]

In [159]:
df.isna().sum()

Transaction ID      0
Date                0
Customer ID         0
Gender              0
Age                 0
Product Category    0
Quantity            0
Price per Unit      0
Total Amount        0
revenue             0
dtype: int64

In [160]:
df.drop_duplicates(inplace=True)

In [161]:
df["month"]=df["Date"].dt.month

In [162]:
df['year']=df["Date"].dt.year

In [163]:
df["revenue"].sum()

456000

In [164]:
df.groupby("Product Category")["revenue"].sum().sort_values(ascending=False)

Product Category
Electronics    156905
Clothing       155580
Beauty         143515
Name: revenue, dtype: int64

In [165]:
df.groupby("Gender")["revenue"].sum().sort_values(ascending=False)

Gender
Female    232840
Male      223160
Name: revenue, dtype: int64

In [166]:
df.groupby("month")["revenue"].sum().sort_values(ascending=False)

month
5     53150
10    46580
12    44690
2     44060
1     36980
8     36960
6     36715
7     35465
11    34920
4     33870
3     28990
9     23620
Name: revenue, dtype: int64

In [167]:
df.groupby('Customer ID')["revenue"].sum().sort_values(ascending=False).head(5)

Customer ID
CUST155    2000
CUST946    2000
CUST269    2000
CUST595    2000
CUST257    2000
Name: revenue, dtype: int64

In [168]:
cust_rev=df.groupby("Customer ID")["revenue"].sum()

In [169]:
q1=cust_rev.quantile(0.25)
q3=cust_rev.quantile(0.75)

In [170]:
high=cust_rev[cust_rev > q3]
medium=cust_rev[cust_rev > q1] & (cust_rev <= q3)
low=cust_rev[cust_rev <= q1]

In [171]:
print("high_spender",len(high))
print("Medium_spender",len(medium))
print("low_spender",len(low))

high_spender 202
Medium_spender 1000
low_spender 262


In [172]:
high_customers=high.index

In [173]:
high_data=df[df["Customer ID"].isin(high_customers)]

In [174]:
high_data["Product Category"].value_counts()

Electronics    70
Beauty         68
Clothing       64
Name: Product Category, dtype: int64

In [175]:
high_data["Gender"].value_counts()

Female    104
Male       98
Name: Gender, dtype: int64

In [176]:
high_data["month"].value_counts()

5     24
2     24
10    22
12    19
4     17
8     16
6     16
1     16
11    14
7     14
3     12
9      8
Name: month, dtype: int64

In [177]:
low_customer=low.index

In [178]:
low_data=df[df["Customer ID"].isin(low_customer)]

In [179]:
low_data["Product Category"].value_counts()

Electronics    98
Clothing       90
Beauty         74
Name: Product Category, dtype: int64

In [180]:
low_data["Gender"].value_counts()

Male      133
Female    129
Name: Gender, dtype: int64

In [181]:
low_data["month"].value_counts()

4     32
8     29
5     28
12    27
6     24
2     23
10    22
11    19
1     19
9     14
7     14
3     11
Name: month, dtype: int64

In [182]:
meadium_customer=medium.index

In [183]:
meadium_data=df[df["Customer ID"].isin(meadium_customer)]

In [184]:
meadium_data["Product Category"].value_counts()

Clothing       351
Electronics    342
Beauty         307
Name: Product Category, dtype: int64

Pareto Analysis 80/20 rule


In [185]:
cust=df.groupby("Customer ID")["revenue"].sum().sort_values(ascending=False)

In [186]:
pareto = cust.reset_index()

In [187]:
pareto.columns=["Customer ID","Total Spend"]

In [188]:
pareto["cum_revenue"]=pareto["Total Spend"].cumsum()

In [189]:
pareto["cum_revenue_%"] = pareto["cum_revenue"] / pareto["Total Spend"].sum() * 100

In [190]:
pareto["cum_customers_%"] = (pareto.index + 1) / len(pareto) * 100

In [191]:
pareto.head(5)

,Customer ID,Total Spend,cum_revenue,cum_revenue_%,cum_customers_%
0,CUST155,2000,2000,0.438596,0.1
1,CUST946,2000,4000,0.877193,0.2
2,CUST269,2000,6000,1.315789,0.3
3,CUST595,2000,8000,1.754386,0.4
4,CUST257,2000,10000,2.192982,0.5


In [192]:
top_20=pareto[pareto["cum_customers_%"]<=20]

In [193]:
top_ids=top_20["Customer ID"]

In [194]:
top_data=df[df["Customer ID"].isin(top_ids)]

In [195]:
top_data["Product Category"].value_counts()


Electronics    70
Beauty         68
Clothing       62
Name: Product Category, dtype: int64

In [196]:
top_data["Gender"].value_counts()

Female    102
Male       98
Name: Gender, dtype: int64

In [197]:
top_data["month"].value_counts()

5     24
2     23
10    22
12    19
4     17
8     16
6     16
1     15
11    14
7     14
3     12
9      8
Name: month, dtype: int64

RFM

What is RFM?

RFM =

R → Recency → How recently customer bought
F → Frequency → How many times they bought
M → Monetary → How much money they spent

In [198]:
rfm=df.groupby("Customer ID").agg({
    "Date":lambda x:(df["Date"].max()-x.max()).days
})

In [199]:
rfm.head(5)

,Date
Customer ID,
CUST001,38
CUST002,308
CUST003,353
CUST004,225
CUST005,240


In [200]:
rfm["Frequency"]=df.groupby("Customer ID")["Transaction ID"].count()

In [201]:
rfm["Monetary"] = df.groupby("Customer ID")["revenue"].sum()

In [203]:
rfm.head(5)

,Date,Frequency,Monetary
Customer ID,,,
CUST001,38,1,150
CUST002,308,1,1000
CUST003,353,1,30
CUST004,225,1,500
CUST005,240,1,100


In [204]:
rfm["R_score"]=pd.qcut(rfm["Date"],5,labels=[5,4,3,2,1])

In [210]:
rfm["F_score"]=1

In [207]:
rfm["M_score"] = pd.qcut(rfm["Monetary"], 5, labels=[1,2,3,4,5])

In [211]:
rfm["RFM_score"] = (
    rfm["R_score"].astype(str) +
    rfm["F_score"].astype(str) +
    rfm["M_score"].astype(str)
)

In [214]:
rfm.max()

Date          365
Frequency       1
Monetary     2000
R_score         1
M_score         5
F_score         1
RFM_score     515
dtype: object

In [215]:
rfm.sort_values("RFM_score", ascending=False).head()

,Date,Frequency,Monetary,R_score,M_score,F_score,RFM_score
Customer ID,,,,,,,
CUST115,36,1,1500,5,5,1,515
CUST805,3,1,1500,5,5,1,515
CUST369,47,1,1500,5,5,1,515
CUST405,56,1,1200,5,5,1,515
CUST757,7,1,1200,5,5,1,515


In [216]:
rfm[rfm["RFM_score"] == rfm["RFM_score"].max()]

,Date,Frequency,Monetary,R_score,M_score,F_score,RFM_score
Customer ID,,,,,,,
CUST047,56,1,1500,5,5,1,515
CUST058,49,1,1200,5,5,1,515
CUST065,27,1,2000,5,5,1,515
CUST074,40,1,2000,5,5,1,515
CUST099,15,1,1200,5,5,1,515
CUST112,30,1,1500,5,5,1,515
CUST115,36,1,1500,5,5,1,515
CUST124,66,1,2000,5,5,1,515
CUST139,17,1,2000,5,5,1,515


In [217]:
def segment(row):
    if row["R_score"] == 5 and row["M_score"] == 5:
        return "High Value Customers"
    
    elif row["R_score"] >= 4 and row["F_score"] >= 3:
        return "Loyal Customers"
    
    elif row["R_score"] <= 2:
        return "Lost Customers"
    
    else:
        return "Average Customers"

rfm["Segment"] = rfm.apply(segment, axis=1)

In [218]:
rfm["Segment"].value_counts()

Average Customers       567
Lost Customers          399
High Value Customers     34
Name: Segment, dtype: int64